In [68]:
import subprocess, os
os.chdir("/workspaces/meow-decoder")
r = subprocess.run(["git", "push", "origin", "main"], capture_output=True, text=True, timeout=60)
print("STDOUT:", r.stdout)
print("STDERR:", r.stderr)
print("RC:", r.returncode)

STDOUT: 
STDERR: To https://github.com/systemslibrarian/meow-decoder
   b24df10..20d2e0e  main -> main

RC: 0


In [ ]:
import subprocess, os
os.chdir("/workspaces/meow-decoder")
os.environ["MEOW_TEST_MODE"] = "1"

r = subprocess.run(
    ["python3", "-m", "pytest", "tests/", "-q", "--tb=line", "-x"],
    capture_output=True, text=True, timeout=300
)
lines = r.stdout.split('\n')
print('\n'.join(lines[-40:]))
if r.returncode != 0 and r.stderr:
    print("STDERR:", r.stderr[-500:])

Rust: test result: ok. 72 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.13s
test result: ok. 29 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.53s
test result: ok. 90 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.54s
test result: ok. 19 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 9.57s
test result: ok. 74 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.01s
test result: ok. 14 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 3.00s
test result: ok. 23 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 13.17s
test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s
------------------------------------------------------------------------------------
TOTAL                                     5588   4960   1656     75    11%
Coverage HTML written to dir htmlcov
============================= 126 pas

In [8]:
import subprocess, os
os.chdir("/workspaces/meow-decoder")
os.environ["MEOW_TEST_MODE"] = "1"

r = subprocess.run(
    ["python3", "-m", "pytest", "tests/test_stego_phase0.py", "-v", "--tb=short", "-x"],
    capture_output=True, text=True, timeout=120
)
# Print just the last part of the output
lines = r.stdout.split('\n')
# Find test results section
for i, line in enumerate(lines):
    if 'PASSED' in line or 'FAILED' in line or 'ERROR' in line or 'test_' in line or 'ERRORS' in line or '=====' in line:
        break
print('\n'.join(lines[max(0,i-2):]))
if r.returncode != 0:
    print("STDERR:", r.stderr[-3000:])

BUILD: OK
IMPORT: OK - all Phase 0 classes available
  CHANNEL_DISPOSAL=0x4, CHANNEL_COMMENT=0x5


In [2]:
import subprocess, os, sys

# Phase 1 Step 0: Verify environment and imports
env = dict(os.environ, MEOW_TEST_MODE="1", PYTHONPATH="/workspaces/meow-decoder")
r = subprocess.run(
    [sys.executable, "-c", """
import sys
print(f"Python: {sys.version}")

# Test all required imports
from meow_decoder.stego_multilayer import (
    MultiLayerStegoEncoder, MultiLayerStegoDecoder,
    MultiLayerConfig, validate_stego, CoercionLevel,
    ADVERSARIAL_STRENGTH_OFF, ADVERSARIAL_STRENGTH_LOW,
    ADVERSARIAL_STRENGTH_MEDIUM, ADVERSARIAL_STRENGTH_HIGH,
)
print("stego_multilayer imports OK")

import imageio.v3 as iio
print("imageio OK")

import numpy as np
from PIL import Image
print("numpy + PIL OK")

# Check carriers
import glob
carriers = sorted(glob.glob("/workspaces/meow-decoder/assets/cat*.jpg"))
print(f"Carriers found: {carriers}")

# Check Rust backend
try:
    import meow_crypto_rs
    print(f"meow_crypto_rs OK - has stego: {hasattr(meow_crypto_rs, 'stego_derive_frame_seed')}")
except ImportError:
    print("meow_crypto_rs NOT available (Python fallback)")
"""],
    env=env, capture_output=True, text=True, timeout=30
)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr[-500:])

Python: 3.11.14 (main, Jan 13 2026, 06:08:32) [GCC 12.2.0]
stego_multilayer imports OK
imageio OK
numpy + PIL OK
Carriers found: ['/workspaces/meow-decoder/assets/cat1.jpg', '/workspaces/meow-decoder/assets/cat2.jpg', '/workspaces/meow-decoder/assets/cat3.jpg', '/workspaces/meow-decoder/assets/cat4.jpg', '/workspaces/meow-decoder/assets/cat5.jpg']
meow_crypto_rs OK - has stego: True



In [1]:
print("kernel alive")

kernel alive


In [ ]:
import subprocess, os, sys
env = dict(os.environ, MEOW_TEST_MODE="1", PYTHONPATH="/workspaces/meow-decoder")
r = subprocess.run(
    [sys.executable, "/workspaces/meow-decoder/_audit_phase1_generate.py"],
    env=env, capture_output=True, text=True, timeout=600, cwd="/workspaces/meow-decoder"
)
print(r.stdout[-8000:] if len(r.stdout) > 8000 else r.stdout)
if r.returncode != 0:
    print(f"\nRETURN CODE: {r.returncode}")
    print("STDERR:", r.stderr[-3000:])

In [1]:
import subprocess, os, sys
env = dict(os.environ, MEOW_TEST_MODE="1", PYTHONPATH="/workspaces/meow-decoder")
# Quick single-artifact test
r = subprocess.run(
    [sys.executable, "-c", """
import os, hashlib, numpy as np, traceback
os.environ["MEOW_TEST_MODE"] = "1"
from pathlib import Path
from PIL import Image
import imageio.v3 as iio
from meow_decoder.stego_multilayer import (
    MultiLayerStegoEncoder, MultiLayerStegoDecoder,
    MultiLayerConfig, validate_stego,
)

OUT = Path("/workspaces/meow-decoder/_audit_artifacts")
OUT.mkdir(exist_ok=True)

# Create carrier GIF from cat1.jpg
img = Image.open("/workspaces/meow-decoder/assets/cat1.jpg").convert("RGB").resize((320, 240))
arr = np.array(img)
rng = np.random.RandomState(42)
frames = []
for i in range(10):
    noise = rng.normal(0, 0.5, arr.shape).astype(np.int16)
    frame = np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    frames.append(frame)
cover = OUT / "test_cover.gif"
iio.imwrite(str(cover), np.stack(frames), duration=100, loop=0)
print(f"Cover created: {cover.stat().st_size} bytes")

# Create stego
key = hashlib.sha256(b"test_key").digest()
payload = bytes(rng.randint(0, 256, size=4096, dtype=np.uint8))
config = MultiLayerConfig()
print(f"Config: channels={config.enable_primary},{config.enable_secondary},{config.enable_tertiary},{config.enable_disposal},{config.enable_comment},{config.enable_temporal}")
print(f"  immunize={config.immunize}, adversarial={config.adversarial_strength}, stc={config.use_stc}")

stego_path = OUT / "test_single.gif"
try:
    encoder = MultiLayerStegoEncoder(config, key)
    meta = encoder.encode(payload, str(cover), str(stego_path))
    print(f"Encode OK: {stego_path.stat().st_size} bytes")
    print(f"Meta: {meta}")
except Exception as e:
    traceback.print_exc()
    print(f"ENCODE FAILED: {e}")
    raise

# Validate
try:
    vr = validate_stego(str(stego_path), str(cover))
    print(f"Steganalysis: {vr.summary}")
except Exception as e:
    traceback.print_exc()
    print(f"VALIDATE FAILED: {e}")

# Decode
try:
    decoder = MultiLayerStegoDecoder(config, key)
    result = decoder.decode(str(stego_path))
    print(f"Decode: mac_valid={result.mac_valid}, channels={result.channel_sources}")
    print(f"Roundtrip match: {result.payload_bytes == payload}")
    print(f"Payload sizes - got: {len(result.payload_bytes)}, expected: {len(payload)}")
except Exception as e:
    traceback.print_exc()
    print(f"DECODE FAILED: {e}")
"""],
    env=env, capture_output=True, text=True, timeout=120, cwd="/workspaces/meow-decoder"
)
print(r.stdout)
if r.returncode != 0:
    print(f"RETURN CODE: {r.returncode}")
    print("STDERR:", r.stderr[-3000:])

TimeoutExpired: Command '['/workspaces/meow-decoder/.venv/bin/python', '-c', '\nimport os, hashlib, numpy as np, traceback\nos.environ["MEOW_TEST_MODE"] = "1"\nfrom pathlib import Path\nfrom PIL import Image\nimport imageio.v3 as iio\nfrom meow_decoder.stego_multilayer import (\n    MultiLayerStegoEncoder, MultiLayerStegoDecoder,\n    MultiLayerConfig, validate_stego,\n)\n\nOUT = Path("/workspaces/meow-decoder/_audit_artifacts")\nOUT.mkdir(exist_ok=True)\n\n# Create carrier GIF from cat1.jpg\nimg = Image.open("/workspaces/meow-decoder/assets/cat1.jpg").convert("RGB").resize((320, 240))\narr = np.array(img)\nrng = np.random.RandomState(42)\nframes = []\nfor i in range(10):\n    noise = rng.normal(0, 0.5, arr.shape).astype(np.int16)\n    frame = np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)\n    frames.append(frame)\ncover = OUT / "test_cover.gif"\niio.imwrite(str(cover), np.stack(frames), duration=100, loop=0)\nprint(f"Cover created: {cover.stat().st_size} bytes")\n\n# 

In [1]:
import subprocess, os, sys
env = dict(os.environ, MEOW_TEST_MODE="1", PYTHONPATH="/workspaces/meow-decoder")
r = subprocess.run(
    [sys.executable, "-c", """
import os, hashlib, numpy as np, time, traceback
os.environ["MEOW_TEST_MODE"] = "1"
from pathlib import Path
from PIL import Image
import imageio.v3 as iio
from meow_decoder.stego_multilayer import (
    MultiLayerStegoEncoder, MultiLayerConfig,
)

OUT = Path("/workspaces/meow-decoder/_audit_artifacts")
OUT.mkdir(exist_ok=True)

# Minimal carrier: 3 frames, 160x120
img = Image.open("/workspaces/meow-decoder/assets/cat1.jpg").convert("RGB").resize((160, 120))
arr = np.array(img)
frames = np.stack([arr]*3)
cover = OUT / "mini_cover.gif"
iio.imwrite(str(cover), frames, duration=100, loop=0)
print(f"Cover: {cover.stat().st_size} bytes, 3 frames 160x120")

key = hashlib.sha256(b"test").digest()
payload = b"x" * 64  # tiny

# Test 1: primary only, no STC, no immunize, no adversarial
t0 = time.time()
config = MultiLayerConfig(
    enable_primary=True, enable_secondary=False, enable_tertiary=False,
    enable_disposal=False, enable_comment=False, enable_temporal=False,
    use_stc=False, immunize=False, adversarial_strength=0,
)
encoder = MultiLayerStegoEncoder(config, key)
stego = OUT / "mini_test1.gif"
meta = encoder.encode(payload, str(cover), str(stego))
print(f"Test1 (primary, no STC/immune/adv): {time.time()-t0:.2f}s, meta={meta}")

# Test 2: primary only WITH STC
t0 = time.time()
config2 = MultiLayerConfig(
    enable_primary=True, enable_secondary=False, enable_tertiary=False,
    enable_disposal=False, enable_comment=False, enable_temporal=False,
    use_stc=True, immunize=False, adversarial_strength=0,
)
encoder2 = MultiLayerStegoEncoder(config2, key)
stego2 = OUT / "mini_test2.gif"
meta2 = encoder2.encode(payload, str(cover), str(stego2))
print(f"Test2 (primary+STC): {time.time()-t0:.2f}s, meta={meta2}")

# Test 3: primary + immunize
t0 = time.time()
config3 = MultiLayerConfig(
    enable_primary=True, enable_secondary=False, enable_tertiary=False,
    enable_disposal=False, enable_comment=False, enable_temporal=False,
    use_stc=True, immunize=True, adversarial_strength=0,
)
encoder3 = MultiLayerStegoEncoder(config3, key)
stego3 = OUT / "mini_test3.gif"
meta3 = encoder3.encode(payload, str(cover), str(stego3))
print(f"Test3 (primary+STC+immunize): {time.time()-t0:.2f}s, meta={meta3}")

# Test 4: all channels, full config
t0 = time.time()
config4 = MultiLayerConfig()
encoder4 = MultiLayerStegoEncoder(config4, key)
stego4 = OUT / "mini_test4.gif"
meta4 = encoder4.encode(payload, str(cover), str(stego4))
print(f"Test4 (full 6-chan config): {time.time()-t0:.2f}s, meta={meta4}")

print("DONE")
"""],
    env=env, capture_output=True, text=True, timeout=300, cwd="/workspaces/meow-decoder"
)
print(r.stdout)
if r.returncode != 0:
    print(f"RC: {r.returncode}")
    print("STDERR:", r.stderr[-3000:])

Cover: 18868 bytes, 3 frames 160x120
Test1 (primary, no STC/immune/adv): 0.17s, meta={'channels_used': ['primary'], 'payload_size': 64, 'prepared_size': 86, 'primary_bits': 688, 'secondary_bits': 0, 'tertiary_bits': 0, 'disposal_bits': 0, 'comment_bytes': 0, 'psnr': 70.23274491236029, 'total_capacity': 688, 'immunized': False, 'saliency_costs': True, 'temporal_bits': 0, 'adversarial_strength': 0}
Test2 (primary+STC): 7.69s, meta={'channels_used': ['primary'], 'payload_size': 64, 'prepared_size': 86, 'primary_bits': 688, 'secondary_bits': 0, 'tertiary_bits': 0, 'disposal_bits': 0, 'comment_bytes': 0, 'psnr': 70.05301120224128, 'total_capacity': 688, 'immunized': False, 'saliency_costs': True, 'temporal_bits': 0, 'adversarial_strength': 0}
Test3 (primary+STC+immunize): 7.70s, meta={'channels_used': ['primary'], 'payload_size': 64, 'prepared_size': 86, 'primary_bits': 688, 'secondary_bits': 0, 'tertiary_bits': 0, 'disposal_bits': 0, 'comment_bytes': 0, 'psnr': 70.53674850515404, 'total_ca

In [2]:
import subprocess, os, sys
env = dict(os.environ, MEOW_TEST_MODE="1", PYTHONPATH="/workspaces/meow-decoder")

# Run fast artifacts first: baseline + all non-STC (ids 0, 6-22, 24)
fast_ids = [0] + list(range(6, 23)) + [24]
for aid in fast_ids:
    r = subprocess.run(
        [sys.executable, "/workspaces/meow-decoder/_audit_phase1_batch.py", str(aid)],
        env=env, capture_output=True, text=True, timeout=120, cwd="/workspaces/meow-decoder"
    )
    # Print last 500 chars of output
    out = r.stdout.strip()
    if out:
        lines = out.split('\n')
        for line in lines[:8]:  # First 8 lines per artifact
            print(line)
    if r.returncode != 0:
        print(f"  FAILED (rc={r.returncode}): {r.stderr[-300:]}")
    print()
print("Fast batch complete!")

[0/25] Generating: baseline_plain
  Plain carrier, no stego (control)
  Baseline: FAIL: RS: PASS (p=0.012) | Chi^2: PASS (det=0.000, p=0.0000) | SPA: DETECTED (rate=0.973)

Total artifacts defined: 25

[6/25] Generating: ml_primary_cat1
  Primary LSB only, no STC, 2KB, cat1
  Encoded in 1.7s: 135655 bytes
  PSNR=35.5dB SSIM=0.9973
  Steg: FAIL: RS: PASS (p=0.030) | Chi^2: PASS (det=0.000, p=0.0000) | SPA: DETECTED (rate=0.984)
  Decoded in 1.0s: RT=False MAC=False CH=['primary']
  STATUS: FAIL(RT)


[7/25] Generating: ml_primary_cat3
  Primary LSB only, no STC, 3KB, cat3
  Encoded in 1.5s: 119374 bytes
  PSNR=37.2dB SSIM=0.9992
  Steg: FAIL: RS: PASS (p=0.012) | Chi^2: PASS (det=0.000, p=0.0000) | SPA: DETECTED (rate=0.977)
  Decoded in 0.9s: RT=False MAC=False CH=['primary']
  STATUS: FAIL(RT)


[8/25] Generating: ml_decoy_cat1
  Decoy key, shallow, 1KB, cat1
  Encoded in 1.6s: 135203 bytes
  PSNR=35.5dB SSIM=0.9973
  Steg: FAIL: RS: PASS (p=0.023) | Chi^2: PASS (det=0.000, p=0.0000) 

In [ ]:
import subprocess, os, sys
env = dict(os.environ, MEOW_TEST_MODE="1", PYTHONPATH="/workspaces/meow-decoder")
# Quick single-artifact test with APNG
r = subprocess.run(
    [sys.executable, "-c", """
import os, hashlib, numpy as np, time, traceback
os.environ["MEOW_TEST_MODE"] = "1"
from pathlib import Path
from PIL import Image
import imageio.v3 as iio
from meow_decoder.stego_multilayer import (
    MultiLayerStegoEncoder, MultiLayerStegoDecoder,
    MultiLayerConfig, validate_stego,
)

OUT = Path("/workspaces/meow-decoder/_audit_artifacts")
OUT.mkdir(exist_ok=True)

# Create APNG carrier from cat1.jpg
img = Image.open("/workspaces/meow-decoder/assets/cat1.jpg").convert("RGB").resize((200, 150))
arr = np.array(img)
rng = np.random.RandomState(42)
frames = []
for i in range(5):
    noise = rng.normal(0, 0.5, arr.shape).astype(np.int16)
    frame = np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    frames.append(frame)

cover = OUT / "test_cover.png"
pil_frames = [Image.fromarray(f, "RGB") for f in frames]
pil_frames[0].save(str(cover), format="PNG", save_all=True,
                   append_images=pil_frames[1:], duration=100, loop=0)
print(f"APNG cover: {cover.stat().st_size} bytes")

# Verify APNG preserves pixels
readback = iio.imread(str(cover), index=None)
if isinstance(readback, np.ndarray) and readback.ndim == 4:
    diff = np.abs(readback[0].astype(int) - frames[0].astype(int))
    print(f"APNG pixel preservation: max_diff={diff.max()}, mean_diff={diff.mean():.4f}")

# Encode stego
key = hashlib.sha256(b"test_key").digest()
payload = bytes(rng.randint(0, 256, size=2048, dtype=np.uint8))

# Primary-only, no STC for speed
config = MultiLayerConfig(
    enable_secondary=False, enable_tertiary=False,
    enable_disposal=False, enable_comment=False,
    enable_temporal=False, use_stc=False, immunize=False,
    adversarial_strength=0,
)

stego = OUT / "test_apng_stego.png"
t0 = time.time()
encoder = MultiLayerStegoEncoder(config, key)
meta = encoder.encode(payload, str(cover), str(stego))
print(f"Encode: {time.time()-t0:.2f}s, meta={meta}")
print(f"Stego: {stego.stat().st_size} bytes")

# Verify stego pixels preserved in APNG
stego_read = iio.imread(str(stego), index=None)
if isinstance(stego_read, np.ndarray) and stego_read.ndim == 4:
    print(f"Stego APNG frames: {stego_read.shape}")

# Decode
decoder = MultiLayerStegoDecoder(config, key)
result = decoder.decode(str(stego))
print(f"Decode: mac={result.mac_valid}, channels={result.channel_sources}")
print(f"Roundtrip: {result.payload_bytes == payload}")
if not result.payload_bytes == payload:
    print(f"  payload len: got={len(result.payload_bytes)} expected={len(payload)}")
    # Check first bytes
    got = result.payload_bytes[:20]
    exp = payload[:20]
    print(f"  got[:20] = {got.hex()}")
    print(f"  exp[:20] = {exp.hex()}")

# Steganalysis
vr = validate_stego(str(stego), str(cover))
print(f"Steganalysis: {vr.summary}")
"""],
    env=env, capture_output=True, text=True, timeout=60, cwd="/workspaces/meow-decoder"
)
print(r.stdout)
if r.returncode != 0:
    print(f"RC: {r.returncode}")
    print("STDERR:", r.stderr[-2000:])